In [1]:
import os
from pathlib import Path

import librosa
import numpy as np

project_override = os.environ.get("CARDIOAGENT_PROJECT_ROOT")
if project_override:
    PROJECT_ROOT = Path(project_override).expanduser().resolve()
else:
    PROJECT_ROOT = next(
        (
            candidate.resolve()
            for candidate in (Path.cwd(), *Path.cwd().parents)
            if (candidate / "Data" / "dataset").is_dir()
            and (candidate / "ourdataset" / "train").is_dir()
        ),
        None,
    )

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the CardioAgent-Ops project. Open this notebook from the project, "
        "or set CARDIOAGENT_PROJECT_ROOT to its root directory."
    )

AUDIO_ROOT = PROJECT_ROOT / "ourdataset" / "train"
MEL_ROOT = PROJECT_ROOT / "Data" / "processed" / "features" / "mel_spectrograms"
SPLITS_ROOT = PROJECT_ROOT / "Data" / "processed" / "splits" / "cnn"
MODEL_ROOT = PROJECT_ROOT / "audio_engine" / "artifacts" / "models" / "cnn_custom"
EVALUATION_ROOT = PROJECT_ROOT / "audio_engine" / "evaluation" / "cnn_custom"
REPORT_ROOT = PROJECT_ROOT / "docs" / "report" / "benchmarks" / "cnn_custom"
CLASS_TO_LABEL = {"normal": 0, "abnormal": 1}
FEATURE_VERSION = "mel40_sr2000_v1"

for class_name in CLASS_TO_LABEL:
    class_dir = AUDIO_ROOT / class_name
    if not class_dir.is_dir() or not list(class_dir.glob("*.wav")):
        raise FileNotFoundError(
            f"No WAV files found in {class_dir}. Run the current data-wrangling notebook first."
        )

print(f"Project root: {PROJECT_ROOT}")
print(f"Audio input: {AUDIO_ROOT}")
print(f"Mel features: {MEL_ROOT}")
print(f"Fold manifests: {SPLITS_ROOT}")
print(f"Model output: {MODEL_ROOT}")


Project root: C:\Users\azizb\Downloads\CardioAgent-Ops
Audio input: C:\Users\azizb\Downloads\CardioAgent-Ops\ourdataset\train
Mel features: C:\Users\azizb\Downloads\CardioAgent-Ops\Data\processed\features\mel_spectrograms
Fold manifests: C:\Users\azizb\Downloads\CardioAgent-Ops\Data\processed\splits\cnn
Model output: C:\Users\azizb\Downloads\CardioAgent-Ops\audio_engine\artifacts\models\cnn_custom


In [2]:
import matplotlib.pyplot as plt


def make_mel_image(wav_path, output_path):
    signal, sample_rate = librosa.load(wav_path, sr=2000, mono=True)
    if signal.size == 0:
        raise ValueError(f"Audio file is empty: {wav_path}")

    mel = librosa.feature.melspectrogram(
        y=signal,
        sr=sample_rate,
        n_fft=512,
        hop_length=128,
        n_mels=40,
        fmin=20,
        fmax=1000,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    value_range = float(np.ptp(mel_db))
    image = (
        (mel_db - mel_db.min()) / value_range
        if value_range > 0
        else np.zeros_like(mel_db)
    )
    plt.imsave(output_path, image, cmap="magma", vmin=0, vmax=1)


prepared_counts = {}
for class_name in CLASS_TO_LABEL:
    source_files = sorted((AUDIO_ROOT / class_name).glob("*.wav"))
    output_dir = MEL_ROOT / class_name
    output_dir.mkdir(parents=True, exist_ok=True)
    prepared_count = 0

    for wav_path in source_files:
        image_path = output_dir / f"{wav_path.stem}_{FEATURE_VERSION}.png"
        if image_path.exists() and image_path.stat().st_mtime >= wav_path.stat().st_mtime:
            continue
        make_mel_image(wav_path, image_path)
        prepared_count += 1

    prepared_counts[class_name] = len(list(output_dir.glob(f"*_{FEATURE_VERSION}.png")))
    print(
        f"{class_name}: {prepared_counts[class_name]} spectrograms ready "
        f"({prepared_count} generated this run)"
    )


normal: 1016 spectrograms ready (1016 generated this run)
abnormal: 2769 spectrograms ready (2769 generated this run)


In [3]:
import csv

from sklearn.model_selection import StratifiedKFold

N_SPLITS = 3
records = []
for class_name, label in CLASS_TO_LABEL.items():
    image_files = sorted((MEL_ROOT / class_name).glob(f"*_{FEATURE_VERSION}.png"))
    if not image_files:
        raise FileNotFoundError(
            f"No spectrogram PNGs found for {class_name} in {MEL_ROOT / class_name}. "
            "Run the spectrogram-preparation cell first."
        )
    records.extend((image_path, label, class_name) for image_path in image_files)

labels = np.asarray([record[1] for record in records], dtype=np.int32)
class_counts = np.bincount(labels, minlength=len(CLASS_TO_LABEL))
if int(class_counts.min()) < N_SPLITS:
    raise ValueError(
        f"Each class needs at least {N_SPLITS} images for stratified folds; "
        f"found {dict(zip(CLASS_TO_LABEL, class_counts.tolist()))}."
    )

SPLITS_ROOT.mkdir(parents=True, exist_ok=True)
splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=10)
for fold_number, (train_indices, test_indices) in enumerate(
    splitter.split(np.zeros(len(records)), labels), start=1
):
    split_path = SPLITS_ROOT / f"cnn_fold_{fold_number}.csv"
    with split_path.open("w", encoding="utf-8", newline="") as split_file:
        writer = csv.DictWriter(
            split_file,
            fieldnames=("split", "relative_path", "label", "class_name"),
        )
        writer.writeheader()
        for split_name, indices in (("train", train_indices), ("test", test_indices)):
            for index in indices:
                image_path, label, class_name = records[int(index)]
                writer.writerow(
                    {
                        "split": split_name,
                        "relative_path": image_path.relative_to(PROJECT_ROOT).as_posix(),
                        "label": int(label),
                        "class_name": class_name,
                    }
                )
    print(f"Fold {fold_number}: {split_path}")


Fold 1: C:\Users\azizb\Downloads\CardioAgent-Ops\Data\processed\splits\cnn\cnn_fold_1.csv
Fold 2: C:\Users\azizb\Downloads\CardioAgent-Ops\Data\processed\splits\cnn\cnn_fold_2.csv
Fold 3: C:\Users\azizb\Downloads\CardioAgent-Ops\Data\processed\splits\cnn\cnn_fold_3.csv


In [4]:
import librosa
import sklearn
import tensorflow as tf

print(f"TensorFlow: {tf.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"librosa: {librosa.__version__}")
print("Runtime: local project paths; no Google Drive mount required.")


TensorFlow: 2.21.0
scikit-learn: 1.9.1
librosa: 1.0.0
Runtime: local project paths; no Google Drive mount required.


In [5]:
import csv

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import callbacks, layers, models, optimizers

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 0.0001


def load_fold_rows(split_csv, split_name):
    image_paths = []
    labels = []
    with split_csv.open("r", encoding="utf-8", newline="") as split_file:
        for row in csv.DictReader(split_file):
            if row["split"] == split_name:
                image_path = (PROJECT_ROOT / row["relative_path"]).resolve()
                if not image_path.is_file():
                    raise FileNotFoundError(f"CNN input image not found: {image_path}")
                image_paths.append(str(image_path))
                labels.append(int(row["label"]))
    if not image_paths:
        raise ValueError(f"No {split_name} images listed in {split_csv}")
    return image_paths, np.asarray(labels, dtype=np.int32)


def make_dataset(image_paths, labels, training):
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    def decode_png(image_path, label):
        image_bytes = tf.io.read_file(image_path)
        image = tf.io.decode_png(image_bytes, channels=3)
        image = tf.image.resize(image, IMAGE_SIZE)
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    dataset = dataset.map(decode_png, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        dataset = dataset.shuffle(
            buffer_size=len(image_paths), seed=10, reshuffle_each_iteration=True
        )
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


def make_model():
    model = models.Sequential(
        [
            layers.Input(shape=(*IMAGE_SIZE, 3)),
            layers.Conv2D(32, 3, padding="same", activation="relu"),
            layers.MaxPooling2D(),
            layers.Conv2D(64, 3, padding="same", activation="relu"),
            layers.MaxPooling2D(),
            layers.Conv2D(32, 3, padding="same", activation="relu"),
            layers.GlobalAveragePooling2D(),
            layers.Dropout(0.4),
            layers.Dense(1, activation="sigmoid"),
        ],
        name="cardio_custom_cnn",
    )
    model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )
    return model


fold_files = sorted(SPLITS_ROOT.glob("cnn_fold_*.csv"))
if len(fold_files) != N_SPLITS:
    raise FileNotFoundError(
        f"Expected {N_SPLITS} fold CSVs in {SPLITS_ROOT}; found {len(fold_files)}. "
        "Run the split-generation cell first."
    )

MODEL_ROOT.mkdir(parents=True, exist_ok=True)
EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
fold_scores = []

for fold_number, split_csv in enumerate(fold_files, start=1):
    train_paths, train_labels = load_fold_rows(split_csv, "train")
    test_paths, test_labels = load_fold_rows(split_csv, "test")
    train_data = make_dataset(train_paths, train_labels, training=True)
    test_data = make_dataset(test_paths, test_labels, training=False)

    tf.keras.backend.clear_session()
    model = make_model()
    fold_model_dir = MODEL_ROOT / f"fold_{fold_number}"
    fold_model_dir.mkdir(parents=True, exist_ok=True)
    best_model_path = fold_model_dir / "best.keras"
    checkpoint = callbacks.ModelCheckpoint(
        filepath=str(best_model_path),
        monitor="val_recall",
        mode="max",
        save_best_only=True,
        verbose=1,
    )
    early_stopping = callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    )

    print(f"Training fold {fold_number}: {len(train_paths)} train, {len(test_paths)} test")
    history = model.fit(
        train_data,
        validation_data=test_data,
        epochs=EPOCHS,
        callbacks=[checkpoint, early_stopping],
        verbose=1,
    )
    best_model = tf.keras.models.load_model(best_model_path)
    scores = best_model.evaluate(test_data, return_dict=True, verbose=0)
    probabilities = best_model.predict(test_data, verbose=0).reshape(-1)
    predictions = (probabilities >= 0.5).astype(np.int32)
    report = classification_report(
        test_labels,
        predictions,
        labels=[0, 1],
        target_names=["normal", "abnormal"],
        zero_division=0,
    )
    matrix = confusion_matrix(test_labels, predictions, labels=[0, 1])
    fold_scores.append(scores)

    report_path = EVALUATION_ROOT / f"fold_{fold_number}_report.txt"
    report_path.write_text(
        f"Metrics: {scores}\nConfusion matrix (normal, abnormal):\n{matrix}\n\n{report}",
        encoding="utf-8",
    )
    figure, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(history.history["accuracy"], label="train")
    axes[0].plot(history.history["val_accuracy"], label="test")
    axes[0].set_title(f"Fold {fold_number} accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[1].plot(history.history["loss"], label="train")
    axes[1].plot(history.history["val_loss"], label="test")
    axes[1].set_title(f"Fold {fold_number} loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    figure.tight_layout()
    figure.savefig(REPORT_ROOT / f"fold_{fold_number}_training.png", dpi=150)
    plt.close(figure)

    print(f"Fold {fold_number} metrics: {scores}")
    print(f"Confusion matrix (normal, abnormal):\n{matrix}")
    print(report)

print("Mean fold accuracy:", float(np.mean([score["accuracy"] for score in fold_scores])))
print("Mean abnormal recall:", float(np.mean([score["recall"] for score in fold_scores])))



Training fold 1: 2523 train, 1262 test
Epoch 1/10


c:\Users\azizb\Downloads\CardioAgent-Ops\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - accuracy: 0.7281 - loss: 0.6146 - precision: 0.7326 - recall: 0.9897
Epoch 1: val_recall improved from None to 1.00000, saving model to C:\Users\azizb\Downloads\CardioAgent-Ops\audio_engine\artifacts\models\cnn_custom\fold_1\best.keras

Epoch 1: finished saving model to C:\Users\azizb\Downloads\CardioAgent-Ops\audio_engine\artifacts\models\cnn_custom\fold_1\best.keras
158/158 ━━━━━━━━━━━━━━━━━━━━ 28s 131ms/step - accuracy: 0.7281 - loss: 0.6146 - precision: 0.7326 - recall: 0.9897 - val_accuracy: 0.7314 - val_loss: 0.5765 - val_precision: 0.7314 - val_recall: 1.0000
Epoch 2/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 0.7329 - loss: 0.5514 - precision: 0.7355 - recall: 0.9913
Epoch 2: val_recall did not improve from 1.00000
158/158 ━━━━━━━━━━━━━━━━━━━━ 21s 132ms/step - accuracy: 0.7329 - loss: 0.5514 - precision: 0.7355 - recall: 0.9913 - val_accuracy: 0.7290 - val_loss: 0.5119 - val_precision: 0.7348 - val_recall: 0.9848
Epoch 

c:\Users\azizb\Downloads\CardioAgent-Ops\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - accuracy: 0.7317 - loss: 0.6023 - precision: 0.7317 - recall: 1.0000
Epoch 1: val_recall improved from None to 1.00000, saving model to C:\Users\azizb\Downloads\CardioAgent-Ops\audio_engine\artifacts\models\cnn_custom\fold_2\best.keras

Epoch 1: finished saving model to C:\Users\azizb\Downloads\CardioAgent-Ops\audio_engine\artifacts\models\cnn_custom\fold_2\best.keras
158/158 ━━━━━━━━━━━━━━━━━━━━ 22s 130ms/step - accuracy: 0.7317 - loss: 0.6023 - precision: 0.7317 - recall: 1.0000 - val_accuracy: 0.7314 - val_loss: 0.5704 - val_precision: 0.7314 - val_recall: 1.0000
Epoch 2/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - accuracy: 0.7388 - loss: 0.5380 - precision: 0.7431 - recall: 0.9827
Epoch 2: val_recall did not improve from 1.00000
158/158 ━━━━━━━━━━━━━━━━━━━━ 21s 128ms/step - accuracy: 0.7388 - loss: 0.5380 - precision: 0.7431 - recall: 0.9827 - val_accuracy: 0.7417 - val_loss: 0.5118 - val_precision: 0.7682 - val_recall: 0.9263
Epoch 

c:\Users\azizb\Downloads\CardioAgent-Ops\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - accuracy: 0.7112 - loss: 0.6143 - precision: 0.7348 - recall: 0.9469
Epoch 1: val_recall improved from None to 1.00000, saving model to C:\Users\azizb\Downloads\CardioAgent-Ops\audio_engine\artifacts\models\cnn_custom\fold_3\best.keras

Epoch 1: finished saving model to C:\Users\azizb\Downloads\CardioAgent-Ops\audio_engine\artifacts\models\cnn_custom\fold_3\best.keras
158/158 ━━━━━━━━━━━━━━━━━━━━ 22s 130ms/step - accuracy: 0.7112 - loss: 0.6143 - precision: 0.7348 - recall: 0.9469 - val_accuracy: 0.7320 - val_loss: 0.5598 - val_precision: 0.7320 - val_recall: 1.0000
Epoch 2/10
158/158 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.7345 - loss: 0.5385 - precision: 0.7402 - recall: 0.9816
Epoch 2: val_recall did not improve from 1.00000
158/158 ━━━━━━━━━━━━━━━━━━━━ 20s 125ms/step - accuracy: 0.7345 - loss: 0.5385 - precision: 0.7402 - recall: 0.9816 - val_accuracy: 0.7502 - val_loss: 0.5066 - val_precision: 0.7667 - val_recall: 0.9469
Epoch 

In [6]:
# ---- MLflow (Service 6) ----
# Cell 4 creates the per-fold checkpoints, reports, and training curves used here.
import re
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("audio_benchmark")
mlflow.set_system_metrics_sampling_interval(1)
mlflow.enable_system_metrics_logging()

if len(fold_scores) != N_SPLITS:
    raise ValueError(f"Expected {N_SPLITS} fold scores, found {len(fold_scores)}. Run Cell 4 first.")

fold_records = []
for fold_number, scores in enumerate(fold_scores, start=1):
    report_path = EVALUATION_ROOT / f"fold_{fold_number}_report.txt"
    curve_path = REPORT_ROOT / f"fold_{fold_number}_training.png"
    model_path = MODEL_ROOT / f"fold_{fold_number}" / "best.keras"
    for required_path in (report_path, curve_path, model_path):
        if not required_path.is_file():
            raise FileNotFoundError(f"Required fold artifact not found: {required_path}")
    report_text = report_path.read_text(encoding="utf-8")
    matrix_text = report_text.split("Confusion matrix (normal, abnormal):\n", 1)[1].split("\n\n", 1)[0]
    values = [int(value) for value in re.findall(r"\d+", matrix_text)]
    if len(values) != 4:
        raise ValueError(f"Could not read a 2x2 confusion matrix from {report_path}")
    metrics = {name: float(scores[name]) for name in ("accuracy", "precision", "recall")}
    metrics["f1"] = 0.0 if metrics["precision"] + metrics["recall"] == 0 else 2 * metrics["precision"] * metrics["recall"] / (metrics["precision"] + metrics["recall"])
    fold_records.append({
        "fold": fold_number, "metrics": metrics, "matrix": np.asarray(values).reshape(2, 2).tolist(),
        "report_path": report_path, "curve_path": curve_path, "model_path": model_path,
    })

with mlflow.start_run(run_name="cnn_custom", log_system_metrics=True) as run:
    mlflow.set_tags({
        "model": "custom_cnn", "task": "pcg_normal_vs_abnormal",
        "evaluation": "3_fold_cross_validation", "feature_set": FEATURE_VERSION,
    })
    mlflow.log_params({
        "n_folds": N_SPLITS, "epochs_requested": EPOCHS, "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE, "image_size": "224x224", "mel_bands": 40, "sample_rate": 2000,
    })
    for metric in ("accuracy", "precision", "recall", "f1"):
        mlflow.log_metric(f"mean_{metric}", float(np.mean([record["metrics"][metric] for record in fold_records])))
    for record in fold_records:
        for metric, value in record["metrics"].items():
            mlflow.log_metric(f"fold_{metric}", value, step=record["fold"])
        mlflow.log_artifact(str(record["report_path"]), artifact_path=f"reports/fold_{record['fold']}")
        mlflow.log_artifact(str(record["curve_path"]), artifact_path=f"figures/fold_{record['fold']}")
    mlflow.log_dict({"folds": [{"fold": record["fold"], **record["metrics"]} for record in fold_records]}, "reports/cv_metrics.json")
    mlflow.log_dict({"labels": ["normal", "abnormal"], "folds": [{"fold": record["fold"], "matrix": record["matrix"]} for record in fold_records]}, "reports/confusion_matrices.json")
    best_fold = max(fold_records, key=lambda record: (record["metrics"]["recall"], record["metrics"]["f1"]))
    mlflow.set_tag("best_fold", str(best_fold["fold"]))
    mlflow.log_artifact(str(best_fold["model_path"]), artifact_path="models/best_fold")
    print(f"Logged CNN run: {run.info.run_id}; best fold: {best_fold['fold']}")


2026/11/14 09:49:25 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/11/14 09:49:25 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/11/14 09:49:25 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/11/14 09:49:25 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


Logged CNN run: c17147e3bb564ff086fa332e581440c0; best fold: 3
🏃 View run cnn_custom at: http://localhost:5000/#/experiments/1/runs/c17147e3bb564ff086fa332e581440c0
🧪 View experiment at: http://localhost:5000/#/experiments/1
